In [1]:
# Test libraries
import requests, bs4, lxml, pandas
print("All libraries successfully imported!")


All libraries successfully imported!


In [4]:
import requests

url = "https://towardsdatascience.com/"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

response = requests.get(url, headers=headers)

print("--- Initiating Contact ---")
print(f"Status Code: {response.status_code}")

print("HTML Content Preview (First 500 characters):")
print(response.text[:500])
print("---")

--- Initiating Contact ---
Status Code: 200
HTML Content Preview (First 500 characters):
<!DOCTYPE html>
<html lang="en-US">
<head>
	<meta charset="UTF-8" />
	<script src="https://h030.towardsdatascience.com/script.js"></script><!-- Google Tag Manager -->
<script>
	(function (w, d, s, l, i) {
		w[l] = w[l] || [];
		w[l].push({
			'gtm.start': new Date().getTime(),
			event: 'gtm.js'
		});
		var f = d.getElementsByTagName(s)[0],
			j = d.createElement(s),
			dl = l != 'dataLayer' ? '&l=' + l : '';
		j.async = true;
		j.src =
			'https://www.googletagmanager.com/gtm.js?id=' + i + dl;

---


In [ ]:
import requests
from bs4 import BeautifulSoup


url = "https://towardsdatascience.com/"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Referer': 'https://www.google.com/',
    'DNT': '1', # Do Not Track request header
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1'
}

try:  
    response = requests.get(url, headers=headers, timeout=10)
 
    response.raise_for_status()
    
    html_content = response.text
    soup = BeautifulSoup(html_content, 'lxml')

    print("--- Parsing HTML Content ---")
    if soup.title:
        print(f"Parsed Page Title: {soup.title.text.strip()}")
    else:
        print("Page title not found. HTML parsing might need adjustment.")
    print("---")


    print("\nRecent Headlines:")
    headlines = soup.find_all(['h2', 'h3'], limit=5)
    for i, title in enumerate(headlines, 1):
        print(f"{i}. {title.get_text(strip=True)}")

except requests.exceptions.HTTPError as http_err:
    print(f"HTTP error occurred: {http_err}")
except Exception as err:
    print(f"An error occurred: {err}")

--- Parsing HTML Content ---
Parsed Page Title: Towards Data Science
---

Recent Headlines:
1. Playing Connect Four with Deep Q-Learning
2. How AI Tools Generate Technical Debt in IoT Systems — and What to Do About It
3. Latest
4. CSPNet Paper Walkthrough: Just Better, No Tradeoffs
5. Inference Scaling (Test-Time Compute): Why Reasoning Models Raise Your Compute Bill


In [7]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://en.wikipedia.org/wiki/Main_Page"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
}

print(f"Attempting to scrape: {url}")

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')
    article_links_data = []

    # In the news
    in_the_news_div = soup.find('div', id='mp-itn')
    if in_the_news_div:
        links = in_the_news_div.find_all('a', href=True)
        for link in links:
            title_text = link.text.strip()
            href = link['href']

            if title_text and href.startswith('/wiki/') and ':' not in href:
                absolute_url = urljoin(url, href)

                if not any(data['url'] == absolute_url for data in article_links_data):
                    article_links_data.append({
                        'title': title_text,
                        'url': absolute_url
                    })

    # On this day
    on_this_day_div = soup.find('div', id='mp-otd')
    if on_this_day_div:
        links = on_this_day_div.find_all('a', href=True)
        for link in links:
            title_text = link.text.strip()
            href = link['href']

            if title_text and href.startswith('/wiki/') and ':' not in href:
                absolute_url = urljoin(url, href)

                if not any(data['url'] == absolute_url for data in article_links_data):
                    article_links_data.append({
                        'title': title_text,
                        'url': absolute_url
                    })

    if article_links_data:
        print("\nExtracted Article Titles and URLs:")
        for article in article_links_data[:10]:
            print(f"- {article['title']}")
            print(f"  {article['url']}")
    else:
        print("No articles found.")

except Exception as e:
    print(f"Error: {e}")

Attempting to scrape: https://en.wikipedia.org/wiki/Main_Page

Extracted Article Titles and URLs:
- Antigua and Barbuda Labour Party
  https://en.wikipedia.org/wiki/Antigua_and_Barbuda_Labour_Party
- Gaston Browne
  https://en.wikipedia.org/wiki/Gaston_Browne
- the general election
  https://en.wikipedia.org/wiki/2026_Antiguan_general_election
- J. Craig Venter
  https://en.wikipedia.org/wiki/J._Craig_Venter
- sequencing of the human genome
  https://en.wikipedia.org/wiki/Human_Genome_Project#Public_versus_private_approaches
- A train crash
  https://en.wikipedia.org/wiki/2026_Bekasi_train_crash
- Jakarta
  https://en.wikipedia.org/wiki/Jakarta
- the London Marathon
  https://en.wikipedia.org/wiki/2026_London_Marathon
- Sabastian Sawe
  https://en.wikipedia.org/wiki/Sabastian_Sawe
- Tigst Assefa
  https://en.wikipedia.org/wiki/Tigst_Assefa


In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://en.wikipedia.org/wiki/List_of_datasets_for_machine_learning_research"
headers = {'User-Agent': 'Mozilla/5.0'}

response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'lxml')

target_table = soup.find('table', class_='wikitable')

if target_table:
    headers = [th.text.strip() for th in target_table.find('tr').find_all('th')]

    table_data = []
    for row in target_table.find_all('tr')[1:6]:
        cells = row.find_all(['td', 'th'])
        table_data.append([cell.text.strip() for cell in cells])

    df = pd.DataFrame(table_data, columns=headers[:len(table_data[0])])
    print(df.head())

                Type                                           Subtypes
0  Specific category  Finance, Economics, Commerce, Societal, Health...
1              Scope  Supranational Union, National, Subnational, Mu...
2           Language  Mandarin Chinese, Spanish, English, Arabic, Hi...
3               Type          Tabular, Graph, Text, Image, Sound, Video
4              Usage                  Training, validating, and testing


In [9]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time

start_url = "http://quotes.toscrape.com/"
current_page_url = start_url

all_quotes = []
page_counter = 0

while page_counter < 3:
    try:
        response = requests.get(current_page_url)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'lxml')

        quotes = soup.find_all('div', class_='quote')

        for q in quotes:
            text = q.find('span', class_='text').text.strip()
            author = q.find('small', class_='author').text.strip()
            all_quotes.append({'text': text, 'author': author})

        next_li = soup.find('li', class_='next')

        if next_li:
            next_url = next_li.find('a')['href']
            current_page_url = urljoin(start_url, next_url)
            page_counter += 1
            time.sleep(1)
        else:
            break

    except Exception as e:
        print(e)
        break

print(all_quotes[:5])

[{'text': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”', 'author': 'Albert Einstein'}, {'text': '“It is our choices, Harry, that show what we truly are, far more than our abilities.”', 'author': 'J.K. Rowling'}, {'text': '“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”', 'author': 'Albert Einstein'}, {'text': '“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”', 'author': 'Jane Austen'}, {'text': "“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”", 'author': 'Marilyn Monroe'}]


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"

headers = {
    'User-Agent': 'EducationalScraper/1.0 (contact: your-email@example.com)'
}

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')
    table = soup.find('table', class_='wikitable')

    records = []
    rows = table.find_all('tr')[1:]

    for row in rows:
        cols = row.find_all('td')
        if len(cols) >= 3:
            software = cols[0].text.strip()
            creator = cols[1].text.strip()
            release = re.sub(r'\[.*?\]', '', cols[2].text.strip())

            records.append({
                'Software': software,
                'Creator': creator,
                'Initial Release': release
            })

    df = pd.DataFrame(records)
    print(df.head())

except requests.exceptions.HTTPError as err:
    print(f"Failed to retrieve data: {err}")

         Software                                            Creator  \
0           BigDL                                  Jason Dai (Intel)   
1           Caffe                Berkeley Vision and Learning Center   
2         Chainer                                 Preferred Networks   
3  Deeplearning4j  Skymind engineering team; Deeplearning4j commu...   
4       DeepSpeed                                          Microsoft   

  Initial Release  
0            2016  
1            2013  
2            2015  
3            2014  
4            2019  


In [13]:
import requests
from bs4 import BeautifulSoup

test_urls = [
    "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "http://httpbin.org/status/404",
    "http://httpbin.org/delay/6"
]

for url in test_urls:
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'lxml')
        title = soup.find('h1')

        if title:
            print(title.text.strip())

    except requests.exceptions.HTTPError as e:
        print("HTTP Error:", e)
    except requests.exceptions.Timeout:
        print("Timeout Error")
    except requests.exceptions.ConnectionError:
        print("Connection Error")
        

HTTP Error: 403 Client Error: Forbidden for url: https://en.wikipedia.org/wiki/Artificial_intelligence
HTTP Error: 404 Client Error: NOT FOUND for url: http://httpbin.org/status/404
Timeout Error


In [15]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import sqlite3

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"

headers = {
    'User-Agent': 'TechStudentBot/1.0 (Contact: your-email@tup.edu.ph)'
}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'lxml')
    table = soup.find('table', class_='wikitable')

    data = []
    for row in table.find_all('tr')[1:15]:
        cols = row.find_all('td')
        if cols:
            data.append([col.text.strip() for col in cols])

    df = pd.DataFrame(data)

    df.to_csv("data.csv", index=False)

    df.to_json("data.json", orient='records', indent=4)

    conn = sqlite3.connect("data.db")
    df.to_sql("table_data", conn, if_exists='replace', index=False)
    conn.close()

    print("Data saved successfully in CSV, JSON, and SQLite formats.")

except requests.exceptions.HTTPError as http_err:
    print(f"HTTP error occurred: {http_err}")
except Exception as err:
    print(f"An unexpected error occurred: {err}")

Data saved successfully in CSV, JSON, and SQLite formats.
